# Planejamento e Reflexão

Planejar e refletir gastam a mesma moeda, que é fazer mais chamadas ao modelo antes de entregar a resposta. Planejar decide a sequência de passos, e refletir submete a uma crítica o que já saiu.

O notebook parte de uma tarefa que o laço não resolve, põe o plano dentro do estado do agente, acrescenta a crítica com rubrica e termina comparando as três rotas sob o mesmo orçamento.

In [ ]:
# No Google Colab, descomente e rode uma vez (Ambiente de execução > GPU).
# !pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

import re
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import torch
from pydantic import BaseModel, Field

from agentkit import LLM, Agent, tool

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm = LLM(MODEL_NAME, device=device, temperature=0.0, max_tokens=250)
print(llm.model)

## A tarefa que falha em um passo

Os três arquivos abaixo simulam relatórios mensais, cada um com um número dentro. A tarefa é somar os três e gravar o total, e ela só termina depois de listar a pasta, ler cada arquivo, fazer a conta e escrever o resultado.

In [ ]:
REPORTS = Path("workspace/reports")
REPORTS.mkdir(parents=True, exist_ok=True)
MONTHS = {"janeiro.txt": 1200, "fevereiro.txt": 950, "marco.txt": 1430}
for name, value in MONTHS.items():
    (REPORTS / name).write_text(f"Relatorio mensal\nTotal de entregas: {value}\n", encoding="utf-8")
print(sorted(path.name for path in REPORTS.iterdir()), "| soma correta:", sum(MONTHS.values()))

In [ ]:
@tool
def list_files() -> str:
    """Lista os arquivos disponíveis na pasta de relatórios."""
    return ", ".join(sorted(path.name for path in REPORTS.iterdir()))


@tool
def read_file(name: str) -> str:
    """Lê um arquivo da pasta de relatórios e devolve o conteúdo."""
    return (REPORTS / name).read_text(encoding="utf-8")


@tool
def write_file(name: str, content: str) -> str:
    """Escreve um arquivo na pasta de relatórios."""
    (REPORTS / name).write_text(content, encoding="utf-8")
    return f"{name} gravado"


TOOLS = [list_files, read_file, write_file]

In [ ]:
TASK = ("Leia o total de entregas de cada arquivo da pasta de relatórios, "
        "some os valores e grave o resultado em total.txt.")

started = time.perf_counter()
direct_messages = Agent(llm, TOOLS, max_steps=6).run(TASK)
direct_seconds = time.perf_counter() - started
for message in direct_messages:
    print(message["role"], ":", message.get("content") or message["tool_calls"])

O agente listou a pasta e parou. Em vez de chamar `read_file` com um dos três nomes que acabaram de voltar, ele pediu ao usuário o caminho dos arquivos, e `total.txt` nunca foi criado.

A listagem já trazia tudo que faltava para o passo seguinte. O que não existe no laço é uma representação do que ainda falta fazer.

## ReAct: pensar, agir, observar

O laço que acabou de rodar alterna raciocínio, chamada de ferramenta e observação, e decide o passo seguinte olhando o que voltou. Esse padrão se chama ReAct, e é o mesmo laço já usado com ferramentas. A tabela abaixo é a execução anterior, vista como sequência de eventos.

In [ ]:
pd.DataFrame([{
    "papel": message["role"],
    "chamadas": message.get("tool_calls", ""),
    "conteúdo": (message.get("content") or "")[:60],
} for message in direct_messages])

Quatro linhas para uma tarefa de quatro etapas, e a execução terminou depois da primeira. O laço decide bem o passo imediato, e nada no histórico registra que três etapas continuam pendentes.

### Exercício 1

Rode o mesmo laço em um pedido que nomeie o arquivo, como ler o total de entregas de `janeiro.txt`, e monte a mesma tabela. Responda quantas chamadas de ferramenta cada execução consumiu e em qual delas o modelo esperou a observação antes de decidir a chamada seguinte.

In [ ]:
# Seu código aqui

## Plano como estrutura

No ReAct o próximo passo é decidido a cada volta. A alternativa é escrever a sequência antes de agir e guardá-la no estado, que é um dicionário com a tarefa, o plano, as observações, o passo atual e o contador de chamadas.

In [ ]:
def new_state(task: str) -> dict:
    """Cria o estado da execução, com o plano e o que já foi observado."""
    return {"task": task, "plan": [], "observations": [], "step": 0, "calls": 0}

In [ ]:
class Plan(BaseModel):
    steps: list[str] = Field(min_length=2, max_length=4)


EXAMPLE = """Exemplo para outra tarefa.
Tarefa: descobrir quantas fotos existem na pasta e abrir a mais antiga.
Passos:
1. Listar os arquivos da pasta.
2. Abrir o arquivo mais antigo da listagem."""

In [ ]:
def make_plan(llm, task: str, tools: list) -> list[str]:
    """Pede ao modelo um plano de dois a quatro passos, validado pelo esquema."""
    available = "\n".join(f"- {fn.tool_schema['name']}: {fn.tool_schema['description']}" for fn in tools)
    return llm.generate_structured([{"role": "user", "content": (
        f"{EXAMPLE}\n\nAgora escreva os passos para a tarefa abaixo, em português, "
        f"cada passo em uma frase.\n\nFerramentas:\n{available}\nTarefa: {task}"
    )}], Plan, max_tokens=200).steps


guessed = make_plan(llm, TASK, TOOLS)
pd.DataFrame({"passo": guessed})

O plano existe antes de qualquer execução e pode ser lido pelo programa, que é a diferença prática entre plano como estrutura e plano como parágrafo. Ele também está errado: o segundo passo esconde três chamadas dentro de um passo só, o terceiro não corresponde a ferramenta nenhuma, e a soma e a gravação não aparecem.

### Plano derivado de uma observação

A quantidade de passos depende de um dado que ninguém observou ainda, que é quantos arquivos existem na pasta. Como o plano é uma estrutura, ele pode ser montado por código a partir da listagem, com um passo por arquivo.

In [ ]:
def plan_from_files() -> list[dict]:
    """Monta o plano a partir da listagem da pasta, com um passo por arquivo."""
    names = [name.strip() for name in list_files().split(",")]
    return [
        {"id": index, "action": f"Leia o arquivo {name} e informe o total de entregas.", "status": "pending"}
        for index, name in enumerate(names, start=1)
    ]


state = new_state(TASK)
state["plan"] = plan_from_files()
pd.DataFrame(state["plan"])

Cada passo agora é uma instrução que uma chamada de ferramenta resolve, e a quantidade de passos veio do mundo em lugar do palpite do modelo. A divisão de responsabilidade é a decisão desta parte: o modelo decide o que fazer, e o código decide quantas vezes.

### O laço que avança o plano

Executar o plano é avançar um campo do estado. Depois de cada observação o agente decide se o plano ainda serve, e refazer o que falta parte da listagem atual da pasta. O replanejamento tem limite, porque um agente que sempre aceita replanejar nunca termina.

In [ ]:
class Decision(BaseModel):
    action: Literal["continuar", "replanejar"]


def should_replan(llm, state: dict) -> bool:
    """Pergunta ao modelo se o plano ainda serve, depois da última observação."""
    state["calls"] += 1
    return llm.generate_structured([{"role": "user", "content": (
        f"Tarefa: {state['task']}\nPasso executado: {state['plan'][state['step'] - 1]['action']}\n"
        f"Observação: {state['observations'][-1]['result']}\n\nO plano ainda serve?"
    )}], Decision, max_tokens=30).action == "replanejar"

In [ ]:
def replan(state: dict) -> list[dict]:
    """Refaz os passos que faltam a partir dos arquivos que existem agora."""
    done = {item["id"] for item in state["observations"]}
    return state["plan"][: state["step"]] + [
        step for step in plan_from_files() if step["id"] not in done
    ]

In [ ]:
def run_plan(llm, state: dict, tools: list, max_replans: int = 1) -> dict:
    """Avança o plano passo a passo e replaneja no máximo max_replans vezes."""
    replans = 0
    while state["step"] < len(state["plan"]):
        step = state["plan"][state["step"]]
        messages = Agent(llm, tools, max_steps=4).run(step["action"])
        step["status"] = "done"
        state["observations"].append({"id": step["id"], "result": (messages[-1]["content"] or "").strip()})
        state["calls"] += sum(1 for message in messages if message["role"] == "assistant")
        state["step"] += 1
        if replans < max_replans and should_replan(llm, state):
            state["plan"] = replan(state)
            replans += 1
    return state

In [ ]:
started = time.perf_counter()
run_plan(llm, state, TOOLS)
planned_seconds = time.perf_counter() - started
print(state["calls"], "chamadas ao modelo em", round(planned_seconds, 1), "segundos")
pd.DataFrame(state["observations"])

Os três arquivos foram lidos e cada número saiu do arquivo certo. A terceira observação trocou o nome do arquivo por `marcado.txt` e acrescentou uma frase que não está no texto, e mesmo assim o total que ela informa está correto.

## Replanejamento

Plano fixo pressupõe que o mundo não muda durante a execução. A célula abaixo remove um arquivo depois de o plano estar montado e roda as duas versões, uma sem replanejamento e outra com uma rodada.

In [ ]:
before = plan_from_files()
(REPORTS / "marco.txt").unlink()
rows = []
for limit in (0, 1):
    stale = new_state(TASK)
    stale["plan"] = [dict(step) for step in before]
    run_plan(llm, stale, TOOLS, max_replans=limit)
    rows.append({"replanejamentos": limit, "passos no plano": len(stale["plan"]),
                 "última observação": stale["observations"][-1]["result"][:60]})
pd.DataFrame(rows)

O plano fixo executou os três passos e o terceiro voltou com o erro de arquivo inexistente, porque a sequência foi montada antes da remoção. Com uma rodada de replanejamento o plano encolheu para dois passos e o passo impossível saiu antes de ser tentado.

In [ ]:
for name, value in MONTHS.items():
    (REPORTS / name).write_text(f"Relatorio mensal\nTotal de entregas: {value}\n", encoding="utf-8")
print(list_files())

## Agregação em Python

As observações trazem os três números escritos em frases, e falta somar. A soma é onde este modelo erra. A agregação dispensa raciocínio e precisa de valores tipados, que é o problema da saída estruturada. Ao modelo fica a parte que só ele faz, que é achar o número dentro de uma frase.

In [ ]:
class Totals(BaseModel):
    values: list[int]


def aggregate(llm, observations: list[dict]) -> list[int]:
    """Extrai um inteiro por observação, para a soma sair de Python."""
    notes = "\n".join(f"Anotação {item['id']}: {item['result']}" for item in observations)
    return llm.generate_structured([{"role": "user", "content": (
        f"Extraia o total de entregas de cada anotação. São {len(observations)} anotações, "
        f"então devolva {len(observations)} números.\n\n{notes}"
    )}], Totals, max_tokens=120).values

In [ ]:
started = time.perf_counter()
values = aggregate(llm, state["observations"])
planned_seconds += time.perf_counter() - started
state["calls"] += 1
planned_answer = f"Total de entregas: {sum(values)}"
print(values, "|", write_file("total.txt", planned_answer + "\n"))
print(read_file("total.txt"))

Os três inteiros saíram validados pelo esquema e a soma foi feita em Python, então `total.txt` recebeu o valor correto. Tirar a aritmética do modelo custa uma chamada e remove do caminho a etapa em que ele erra.

### Exercício 2

Acrescente um quarto relatório à pasta, com o valor que você quiser, e refaça o plano a partir de `plan_from_files`. Responda quantos passos o plano passou a ter e qual total foi gravado em `total.txt`.

In [ ]:
# Seu código aqui

## Reflexão

A segunda técnica trabalha sobre a resposta pronta. Pedir ao modelo que avalie um texto sem critério devolve elogio, então a crítica útil declara a rubrica, sai em estrutura validada e termina em um veredito de conjunto fechado.

In [ ]:
class Critique(BaseModel):
    score: int = Field(ge=0, le=5)
    issues: list[str]
    verdict: Literal["accept", "revise"]


def critique(llm, task: str, answer: str) -> Critique:
    """Avalia uma resposta segundo a rubrica de precisão e completude."""
    return llm.generate_structured([{"role": "user", "content": (
        f"Tarefa: {task}\nResposta: {answer}\n\n"
        "Dê uma nota de 0 a 5 para precisão e completude, liste os problemas "
        "concretos e decida entre accept e revise."
    )}], Critique, max_tokens=250)

In [ ]:
weak_answer = "A soma das entregas dá uns três mil, mais ou menos."
review = critique(llm, TASK, weak_answer)
print(review.score, review.verdict)
for issue in review.issues:
    print(" -", issue)

A nota e a lista de problemas são valores, então o programa pode decidir com eles. Dois dos três apontamentos tratam do enunciado da tarefa em lugar da resposta, e nenhum deles diz que o número está faltando.

### Laço de revisão

A crítica vira laço quando a resposta reescrita volta ao crítico. O veredito de aceitação encerra o laço, e o limite de rodadas encerra quando ele não vem.

In [ ]:
def revise(llm, task: str, answer: str, max_rounds: int = 2) -> dict:
    """Critica e reescreve até o veredito de aceitação ou o fim das rodadas."""
    rounds = []
    for number in range(1, max_rounds + 1):
        review = critique(llm, task, answer)
        rounds.append({"round": number, "score": review.score, "verdict": review.verdict})
        if review.verdict == "accept":
            break
        answer = llm.invoke([{"role": "user", "content": (
            f"Tarefa: {task}\nResposta anterior: {answer}\n"
            f"Problemas encontrados: {review.issues}\n\nEscreva uma resposta melhor."
        )}], max_tokens=150)
    return {"answer": answer, "rounds": rounds}

In [ ]:
revised = revise(llm, TASK, weak_answer)
print(revised["answer"][:250])
print("números na resposta:", re.findall(r"\d+", revised["answer"]))
pd.DataFrame(revised["rounds"])

A nota subiu de 3 para 4, o veredito virou aceitação e a resposta reescrita perdeu o único número que tinha, trocando a estimativa por um roteiro de como a tarefa seria feita. O laço melhorou a nota e piorou a resposta.

Nenhuma das chamadas tem acesso aos arquivos. A crítica redistribui o que já está no contexto e não repõe o dado que falta.

### Exercício 3

Reescreva a rubrica de `critique` para exigir que a resposta contenha o total em algarismos, e rode `revise` de novo sobre `weak_answer`. Responda se o número apareceu e em qual rodada o veredito mudou.

In [ ]:
# Seu código aqui

## Orçamento pareado

As três rotas resolvem a mesma tarefa gastando chamadas de formas diferentes. A autoconsistência gasta amostrando várias respostas e ficando com a mais frequente, e a comparação olha acerto, chamadas ao modelo e tempo.

In [ ]:
def self_consistency(llm, task: str, tools: list, samples: int = 3) -> dict:
    """Roda o laço várias vezes com amostragem e devolve as respostas finais."""
    llm.temperature = 0.8
    answers, calls = [], 0
    for seed in range(samples):
        torch.manual_seed(seed)
        messages = Agent(llm, tools, max_steps=6).run(task)
        answers.append((messages[-1]["content"] or "").strip())
        calls += sum(1 for message in messages if message["role"] == "assistant")
    llm.temperature = 0.0
    return {"answers": answers, "calls": calls}

In [ ]:
started = time.perf_counter()
sampled = self_consistency(llm, TASK, TOOLS)
sampling_seconds = time.perf_counter() - started
for answer in sampled["answers"]:
    print("-", " ".join(answer.split())[:90])

In [ ]:
target = str(sum(MONTHS.values()))
direct_calls = sum(1 for message in direct_messages if message["role"] == "assistant")
pd.DataFrame([
    {"rota": "ReAct direto", "chamadas": direct_calls, "segundos": round(direct_seconds, 1),
     "acertou": target in (direct_messages[-1]["content"] or "")},
    {"rota": "autoconsistência", "chamadas": sampled["calls"], "segundos": round(sampling_seconds, 1),
     "acertou": any(target in answer for answer in sampled["answers"])},
    {"rota": "plano e agregação", "chamadas": state["calls"], "segundos": round(planned_seconds, 1),
     "acertou": target in planned_answer},
])

A rota do plano é a única que acerta, e é também a que faz mais chamadas ao modelo. A autoconsistência roda o laço inteiro três vezes, fica com o maior tempo da tabela e erra nas três amostras, porque nenhuma votação salva um conjunto em que a resposta certa não aparece.

Gastar mais chamadas não decide o resultado. O plano acerta por mudar a estrutura do problema, tirando a soma do modelo e deixando para ele apenas a leitura.

### Exercício 4

Rode as três rotas em uma tarefa de um passo só, como ler o total de entregas de `janeiro.txt`, medindo chamadas e segundos como na tabela acima. Responda qual rota ficou mais cara e o que isso diz sobre quando vale decompor.

In [ ]:
# Seu código aqui